In [1]:
import pandas as pd
import re, html

# Columns
TEXT_COLS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "department",
]

BOOL_COLS = ["telecommuting", "has_company_logo", "has_questions"]

CAT_COLS = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
]

TARGET_COL = "fraudulent"


In [2]:
def clean_text(x: str) -> str:
    if pd.isna(x) or not str(x).strip():
        return "not available"
    s = str(x)
    s = html.unescape(s)
    s = re.sub(r"#URL_[0-9a-f]{10,}#", " URL ", s)
    s = re.sub(r"http\S+", " URL ", s)
    s = re.sub(r"www\.\S+", " URL ", s)
    s = re.sub(r"\S+@\S+", " EMAIL ", s)
    s = s.replace("Â", " ").replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def parse_salary_range(x):
    if pd.isna(x) or not str(x).strip():
        return pd.Series({"salary_min": pd.NA, "salary_max": pd.NA, "salary_avg": pd.NA})
    s = str(x).lower().replace("k", "000")
    nums = re.findall(r"\d[\d,\.]*", s)
    nums = [int(float(n.replace(",", ""))) for n in nums if n]
    if not nums:
        return pd.Series({"salary_min": pd.NA, "salary_max": pd.NA, "salary_avg": pd.NA})
    if len(nums) == 1:
        mn = mx = nums[0]
    else:
        mn, mx = min(nums), max(nums)
    avg = int(round((mn + mx) / 2))
    return pd.Series({"salary_min": mn, "salary_max": mx, "salary_avg": avg})


def split_location(x):
    if pd.isna(x) or not str(x).strip():
        return pd.Series({
            "location_country": "unknown",
            "location_state": "unknown",
            "location_city": "unknown"
        })
    parts = [p.strip() for p in str(x).split(",")]
    while len(parts) < 3:
        parts.append("unknown")
    country, state, city = parts[:3]
    return pd.Series({
        "location_country": country or "unknown",
        "location_state": state or "unknown",
        "location_city": city or "unknown"
    })


def normalize_booleans(df):
    for col in BOOL_COLS:
        if col not in df.columns:
            df[col] = 0
        df[col] = df[col].map(lambda v: 1 if str(v).strip().lower() in ["1", "true", "yes"] else 0)
    return df


def fill_categoricals(df):
    for col in CAT_COLS:
        if col not in df.columns:
            df[col] = "unknown"
        df[col] = df[col].fillna("unknown").astype(str).str.lower().str.strip()
        df[col] = df[col].replace("", "unknown")
    return df


In [3]:
# Load your Kaggle dataset
df = pd.read_csv("fake_job_postings.csv")

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head(3)


Rows: 17880, Columns: 18


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# 1. Clean text columns
for col in TEXT_COLS:
    if col not in df.columns:
        df[col] = "not available"
    df[col] = df[col].apply(clean_text)

# 2. Fill categoricals and booleans
df = fill_categoricals(df)
df = normalize_booleans(df)

# 3. Parse salary_range
if "salary_range" not in df.columns:
    df["salary_range"] = pd.NA
salary_df = df["salary_range"].apply(parse_salary_range)
df = pd.concat([df.drop(columns=["salary_range"]), salary_df], axis=1)

# 4. Split location
if "location" not in df.columns:
    df["location"] = "unknown, unknown, unknown"
loc_df = df["location"].apply(split_location)
df = pd.concat([df, loc_df], axis=1)

# 5. Combine text
df["combined_text"] = df[TEXT_COLS].fillna("not available").agg(" ".join, axis=1)
df["combined_text"] = df["combined_text"].str.replace(r"\s+", " ", regex=True).str.strip()

# 6. Add text stats
df["text_length"] = df["combined_text"].str.len()
df["num_exclamations"] = df["combined_text"].str.count("!")
df["num_urls"] = df["combined_text"].str.count("URL")
df["num_emails"] = df["combined_text"].str.count("EMAIL")

# 7. Target column
if "fraudulent" not in df.columns:
    df["fraudulent"] = pd.NA
else:
    df["fraudulent"] = df["fraudulent"].fillna(0).astype(int).clip(0, 1)

print("✅ Preprocessing complete!")
df.head(3)


✅ Preprocessing complete!


,job_id,title,location,department,company_profile,description,requirements,benefits,telecommuting,has_company_logo,...,salary_max,salary_avg,location_country,location_state,location_city,combined_text,text_length,num_exclamations,num_urls,num_emails
0,1,Marketing Intern,"US, NY, New York",Marketing,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,not available,0,1,...,<NA>,<NA>,US,NY,New York,"Marketing Intern We're Food52, and we've creat...",2681,1,0,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,...,<NA>,<NA>,NZ,unknown,Auckland,Customer Service - Cloud Video Production 90 S...,5702,5,11,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",not available,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,not available,0,1,...,<NA>,<NA>,US,IA,Wever,Commissioning Machinery Assistant (CMA) Valor ...,2662,0,0,0


In [5]:
# Show which columns have missing values
df.isna().mean().sort_values(ascending=False).head(10)


salary_avg         0.839597
salary_min         0.839597
salary_max         0.839597
location           0.019351
company_profile    0.000000
description        0.000000
requirements       0.000000
benefits           0.000000
telecommuting      0.000000
job_id             0.000000
dtype: float64

In [6]:
output_path = "fake_job_postings_clean.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned dataset saved as {output_path}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")


✅ Cleaned dataset saved as fake_job_postings_clean.csv
Rows: 17880, Columns: 28


In [7]:
# Load it back just to confirm
clean_df = pd.read_csv("fake_job_postings_clean.csv")
clean_df.sample(5)[["title", "location_country", "employment_type", "combined_text"]]


,title,location_country,employment_type,combined_text
14655,Furniture Installer $14/Hour,US,full-time,Furniture Installer $14/Hour not available Rec...
12400,Oracle EBS Consultant,US,contract,Oracle EBS Consultant not available Position: ...
7987,"Personal Assistant (Tech, Mobile, Startup)",HK,full-time,"Personal Assistant (Tech, Mobile, Startup) At ..."
16128,President / COO,US,full-time,President / COO Human capital is usually the b...
8192,Sharepoint BI Architect,IN,full-time,Sharepoint BI Architect Visual BI is one of th...


In [8]:
df["fraudulent"].value_counts(normalize=True)

fraudulent
0    0.951566
1    0.048434
Name: proportion, dtype: float64

In [9]:
df["combined_text"].str.len().describe()


count    17880.000000
mean      2630.994743
std       1435.791231
min         79.000000
25%       1591.000000
50%       2493.000000
75%       3441.000000
max      14901.000000
Name: combined_text, dtype: float64

# Model Training

### Baseline

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

df = pd.read_csv("fake_job_postings_clean.csv")

print("Rows:", len(df))
df[["combined_text", "fraudulent"]].head(3)


Rows: 17880


,combined_text,fraudulent
0,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,0


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df["combined_text"],
    df["fraudulent"],
    test_size=0.2,
    random_state=42,
    stratify=df["fraudulent"]
)
print("Train size:", len(X_train), " Test size:", len(X_test))
print("Fraud % in train:", round(y_train.mean()*100,2))


Train size: 14304  Test size: 3576
Fraud % in train: 4.84


In [14]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1,4)   # include unigrams + bigrams
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("Feature matrix shape:", X_train_tfidf.shape)


Feature matrix shape: (14304, 30000)


In [15]:
clf = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",   # handles imbalance
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

clf.fit(X_train_tfidf, y_train)


SGDClassifier(class_weight='balanced', loss='log_loss', random_state=42)

In [16]:
y_pred = clf.predict(X_test_tfidf)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


[[3333   70]
 [  16  157]]
              precision    recall  f1-score   support

           0      0.995     0.979     0.987      3403
           1      0.692     0.908     0.785       173

    accuracy                          0.976      3576
   macro avg      0.843     0.943     0.886      3576
weighted avg      0.981     0.976     0.977      3576



In [17]:
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(clf, "sgd_fraud_model.joblib")
print("✅ Model and vectorizer saved.")


✅ Model and vectorizer saved.


In [18]:
sample = """
Work from home and earn $1500 per week! No experience needed.
Just send your bank details and start today.
"""
sample_vec = vectorizer.transform([sample])
pred = clf.predict(sample_vec)[0]
prob = clf.predict_proba(sample_vec)[0][1]
print("Prediction:", "FAKE" if pred==1 else "REAL", " |  Fake probability:", round(prob,3))


Prediction: FAKE  |  Fake probability: 0.656


### Transformer

In [27]:
!pip install -U "transformers>=4.40" "datasets>=2.18" "accelerate>=0.28" "evaluate>=0.4" -q

  You can safely remove it manually.


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch

C:\Users\Glenn\anaconda3\envs\env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
df = pd.read_csv("fake_job_postings_clean.csv")
df = df[["combined_text", "fraudulent"]].dropna()

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["fraudulent"],
    random_state=42
)

print("Train:", len(train_df), "Val:", len(val_df))

Train: 14304 Val: 3576


In [29]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["combined_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [39]:
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

# Rename label column so Trainer recognizes it
train_ds = train_ds.rename_column("fraudulent", "labels")
val_ds   = val_ds.rename_column("fraudulent", "labels")

# Tokenize
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map: 100%|████████████████████████████████████████████████████████████████| 3576/3576 [00:01<00:00, 2835.15 examples/s]


In [40]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
class_counts = train_df["fraudulent"].value_counts().to_dict()
weight = torch.tensor([
    class_counts[0]/sum(class_counts.values()),
    class_counts[1]/sum(class_counts.values())
])
model.classifier.bias.data = torch.log(weight)

In [42]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./bert_job_scam",
    eval_strategy="epoch",             # <-- use this instead of evaluation_strategy
    save_strategy="epoch",             # keep this one
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    learning_rate=2e-5,
    fp16=True,
    logging_dir="./logs",
    logging_steps=100,
)


In [44]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [48]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,  # (or tokenizer=tokenizer)
    compute_metrics=compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.031000,0.049201,0.990213,0.915663,0.878613,0.896755
2,0.013300,0.065800,0.989933,0.930818,0.855491,0.891566
3,0.013500,0.075130,0.990213,0.925926,0.867052,0.895522


TrainOutput(global_step=5364, training_loss=0.02181917604539574, metrics={'train_runtime': 156.188, 'train_samples_per_second': 274.746, 'train_steps_per_second': 34.343, 'total_flos': 2842220505563136.0, 'train_loss': 0.02181917604539574, 'epoch': 3.0})

In [49]:
preds = trainer.predict(val_ds)
y_true = val_df["fraudulent"].values
y_pred = preds.predictions.argmax(axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.994     0.996     0.995      3403
           1      0.916     0.879     0.897       173

    accuracy                          0.990      3576
   macro avg      0.955     0.937     0.946      3576
weighted avg      0.990     0.990     0.990      3576



In [50]:
trainer.save_model("./distilbert_job_fraud_detector")
tokenizer.save_pretrained("./distilbert_job_fraud_detector")


('./distilbert_job_fraud_detector\\tokenizer_config.json',
 './distilbert_job_fraud_detector\\special_tokens_map.json',
 './distilbert_job_fraud_detector\\vocab.txt',
 './distilbert_job_fraud_detector\\added_tokens.json',
 './distilbert_job_fraud_detector\\tokenizer.json')

In [51]:
def predict_job(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return {"real_prob": float(probs[0]), "fake_prob": float(probs[1])}


In [54]:
sample = """
Software Engineer – Backend (Python)
We’re seeking an experienced backend engineer to design and develop scalable APIs.
Responsibilities include collaborating with cross-functional teams, writing clean code, and maintaining CI/CD pipelines.
Requirements: 3+ years of experience with Python and Django, knowledge of AWS, and a BS in Computer Science or equivalent.
"""
predict_job(sample)

{'real_prob': 0.9999771118164062, 'fake_prob': 2.2872625777381472e-05}